# 🔬 Uncertainty Quantification in Post-Generation Attribution (P-Cite)

## Multi-Model Experiment on ASQA Dataset

This notebook runs the full attribution uncertainty pipeline across **5 LLM models**:

| Model | Organisation | Size |
|-------|-------------|------|
| Llama 3.3 70B Instruct | Meta | 70B |
| Gemma 3 27B IT | Google | 27B |
| Mistral Small 3.1 24B | Mistral | 24B |
| Nemotron 3 Super 120B | NVIDIA | 120B MoE |
| Qwen3 Next 80B | Alibaba | 80B MoE |

**Pipeline per claim:**
1. Generate answer → Extract claims → Generate K=5 perturbations
2. BM25 retrieve top-3 docs for original + each perturbation
3. Embed retrieved docs with `all-MiniLM-L6-v2` (GPU-accelerated)
4. U_attr = Var(cosine_similarities) across perturbations

---

> **GPU Recommended**: Runtime → Change runtime type → **T4 GPU**

## Cell 1: Install Dependencies
Installs all required packages and detects GPU.

In [1]:
# Cell 1: Install Dependencies
print("📦 Installing dependencies ...")
import subprocess, sys

pkgs = [
    "openai>=1.0.0", "python-dotenv", "sentence-transformers",
    "scikit-learn", "numpy", "pandas", "rank-bm25", "datasets", "tqdm",
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

print("✅ All packages installed.")

# Check GPU
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("⚠️  No GPU — embeddings will use CPU (slower but works fine)")
    DEVICE = "cpu"

📦 Installing dependencies ...
✅ All packages installed.
🚀 GPU detected: Tesla T4
   Memory: 15.6 GB


## Cell 2: 🔑 API Key (Secure)
Your key is entered via `getpass` and **never stored in the notebook**.

You can also use Colab Secrets: Settings → Secrets → Add `OPENROUTER_API_KEY`

In [2]:
# Cell 2: API Key (secure — never saved to file)
import getpass, os

API_KEY = None

# Try Colab secrets first
try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
    if API_KEY: print("🔑 API key loaded from Colab Secrets.")
except Exception: pass

# Try Kaggle secrets
if not API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
        if API_KEY: print("🔑 API key loaded from Kaggle Secrets.")
    except Exception: pass

# Manual input as fallback
if not API_KEY:
    API_KEY = getpass.getpass("🔑 Enter your OpenRouter API key: ")

os.environ["OPENROUTER_API_KEY"] = API_KEY
print("✅ API key set (not stored in notebook).")

🔑 Enter your OpenRouter API key: ··········
✅ API key set (not stored in notebook).


## Cell 3: ⚙️ Configuration

**Edit the values below** to control the experiment:
- `NUM_INSTANCES` — how many ASQA questions per model
- `MODELS` — which LLMs to test (comment out any you don't want)
- `USE_ALCE_CORPUS` / `USE_WIKIPEDIA` — toggle data sources
- `PARALLEL_WORKERS` — concurrent API calls (3 = safe for free tier)

In [3]:
# Cell 3: ⚙️ CONFIGURATION — EDIT THIS CELL

# ── How many ASQA questions per model ──
NUM_INSTANCES = 40        # 📌 10=test, 40=standard, 948=full validation

# ── Which models to test (comment/uncomment) ──
MODELS = [
    "meta-llama/llama-3.3-70b-instruct:free",         # Meta — 70B
    "google/gemma-3-27b-it:free",                      # Google — 27B
    "mistralai/mistral-small-3.1-24b-instruct:free",   # Mistral — 24B
    "nvidia/nemotron-3-super-120b-a12b:free",          # NVIDIA — 120B MoE
    "qwen/qwen3-next-80b-a3b-instruct:free",           # Alibaba — 80B MoE
]

MODEL_SHORT = {
    "meta-llama/llama-3.3-70b-instruct:free"       : "llama-3.3-70b",
    "google/gemma-3-27b-it:free"                    : "gemma-3-27b",
    "mistralai/mistral-small-3.1-24b-instruct:free" : "mistral-small-24b",
    "nvidia/nemotron-3-super-120b-a12b:free"        : "nemotron-120b",
    "qwen/qwen3-next-80b-a3b-instruct:free"         : "qwen3-80b",
}

# ── Data Sources ──
USE_ALCE_CORPUS   = True     # 📌 ALCE gold passages (din0s/asqa)
USE_WIKIPEDIA     = True     # 📌 Wikipedia passages (wikimedia/wikipedia)
MAX_WIKI_PASSAGES = 10_000   # Wikipedia passages to stream

# ── Pipeline Settings ──
NUM_PERTURBATIONS = 5        # K perturbations per claim
TOP_K_DOCS        = 3        # docs per BM25 query
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"
RANDOM_SEED       = 42

# ── API Settings ──
MAX_TOKENS_ANSWER  = 512
MAX_TOKENS_PERTURB = 256
TEMPERATURE        = 0.7
MAX_RETRIES        = 6
PARALLEL_WORKERS   = 3       # 📌 concurrent API calls (3 = safe)

# ── Output Directory ──
import pathlib
RESULTS_DIR = pathlib.Path(
    "/content/results" if os.path.exists("/content") else
    "/kaggle/working/results" if os.path.exists("/kaggle") else
    "./results"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Print config summary
print("=" * 60)
print("  EXPERIMENT CONFIGURATION")
print("=" * 60)
print(f"  Instances/model   : {NUM_INSTANCES}")
print(f"  Models            : {len(MODELS)}")
for m in MODELS:
    print(f"    - {MODEL_SHORT.get(m, m)}")
print(f"  Data sources      : {'ALCE ' if USE_ALCE_CORPUS else ''}{' + Wikipedia' if USE_WIKIPEDIA else ''}")
print(f"  Wiki passages     : {MAX_WIKI_PASSAGES:,}")
print(f"  Perturbations     : {NUM_PERTURBATIONS}")
print(f"  Embedding device  : {DEVICE}")
print(f"  Parallel workers  : {PARALLEL_WORKERS}")
print(f"  Output            : {RESULTS_DIR}")
print("=" * 60)

  EXPERIMENT CONFIGURATION
  Instances/model   : 40
  Models            : 5
    - llama-3.3-70b
    - gemma-3-27b
    - mistral-small-24b
    - nemotron-120b
    - qwen3-80b
  Data sources      : ALCE  + Wikipedia
  Wiki passages     : 10,000
  Perturbations     : 5
  Embedding device  : cuda
  Parallel workers  : 3
  Output            : /content/results


## Cell 4: Load Core Modules
All pipeline components: LLM client, claim extractor, BM25, embedder, uncertainty metric.

In [4]:
# Cell 4: Core Modules
import json, logging, random, re, statistics, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

import numpy as np
import pandas as pd
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

logging.basicConfig(level=logging.WARNING, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("pcite")
logger.setLevel(logging.INFO)


class LLMClient:
    """OpenRouter API client with 429-aware retry logic."""

    def __init__(self, model: str):
        self.client = OpenAI(
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
        )
        self.model = model

    def _call(self, messages, max_tokens):
        last_exc = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                r = self.client.chat.completions.create(
                    model=self.model, messages=messages,
                    max_tokens=max_tokens, temperature=TEMPERATURE,
                )
                content = r.choices[0].message.content
                if content is None: raise ValueError("Null content")
                time.sleep(0.5)
                return content.strip()
            except Exception as exc:
                last_exc = exc
                err = str(exc)
                is_rate = "429" in err or "rate" in err.lower() or "402" in err
                wait = min(60.0 * (2 ** (attempt - 1)), 600) if is_rate else 10.0 * attempt
                logger.warning(f"  API {attempt}/{MAX_RETRIES} {'[429]' if is_rate else ''}: retrying in {wait:.0f}s")
                time.sleep(wait)
        raise RuntimeError(f"API failed after {MAX_RETRIES} retries") from last_exc

    def generate_answer(self, question):
        return self._call([
            {"role": "system", "content": "You are a knowledgeable assistant. Answer factual questions accurately in 3-6 sentences."},
            {"role": "user", "content": f"Answer: {question}"},
        ], max_tokens=MAX_TOKENS_ANSWER)

    def generate_one_perturbation(self, claim):
        text = self._call([
            {"role": "system", "content": "Rephrase the factual claim without changing its meaning. Return ONLY the rephrased claim."},
            {"role": "user", "content": f"Rephrase: {claim}"},
        ], max_tokens=MAX_TOKENS_PERTURB)
        for prefix in ("rephrased claim:", "claim:", "rephrase:"):
            if text.lower().startswith(prefix):
                text = text[len(prefix):].strip()
        return text if text else claim


def extract_claims(text):
    """Split text into sentence-level claims."""
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if len(s.strip()) > 15]


class BM25Retriever:
    def __init__(self, passages):
        self.passages = passages
        tokenized = [p["text"].lower().split() for p in passages]
        self.bm25 = BM25Okapi(tokenized)

    def retrieve(self, query, top_k=3):
        scores = self.bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[-top_k:][::-1]
        return [self.passages[i] for i in top_idx]


class Embedder:
    def __init__(self):
        self.model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
        logger.info(f"  Embedding model loaded on {DEVICE}")

    def embed(self, texts):
        return self.model.encode(texts, batch_size=64, show_progress_bar=False, convert_to_numpy=True)

    def mean_embedding(self, texts):
        return self.embed(texts).mean(axis=0)


def cosine_sim(a, b):
    return float(sklearn_cosine(a.reshape(1,-1), b.reshape(1,-1))[0,0])


def compute_uncertainty(sims):
    """U_attr = Var(cosine_similarities)"""
    return float(np.var(sims)) if len(sims) >= 2 else 0.0


print("✅ All modules loaded.")

✅ All modules loaded.


## Cell 5: Load ASQA Dataset
Downloads the ASQA ambiguous QA dataset from HuggingFace.

In [5]:
# Cell 5: Load ASQA Dataset
from datasets import load_dataset

print(f"📚 Loading ASQA dataset (first {NUM_INSTANCES} instances) ...")
try:
    asqa_ds = load_dataset("din0s/asqa", split="dev")
except Exception:
    asqa_ds = load_dataset("din0s/asqa", split="dev", trust_remote_code=True)

asqa_records = list(asqa_ds.select(range(min(NUM_INSTANCES, len(asqa_ds)))))
questions = [r["ambiguous_question"] for r in asqa_records]

print(f"✅ Loaded {len(questions)} questions.")
for i, q in enumerate(questions[:5], 1):
    print(f"   Q{i}: {q}")
if len(questions) > 5:
    print(f"   ... and {len(questions) - 5} more")

📚 Loading ASQA dataset (first 40 instances) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-87b7d64f7913b5(…):   0%|          | 0.00/5.27M [00:00<?, ?B/s]

data/dev-00000-of-00001-58a9a40c6e69f07b(…):   0%|          | 0.00/1.46M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4353 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/948 [00:00<?, ? examples/s]

✅ Loaded 40 questions.
   Q1: Who has the highest goals in world football?
   Q2: Who is the original artist of sound of silence?
   Q3: When was the first apple i phone made?
   Q4: Who played the weasley brothers in harry potter?
   Q5: How many state parks are there in virginia?
   ... and 35 more


## Cell 6: Load Retrieval Corpus
Loads ALCE gold passages and/or Wikipedia passages based on your configuration.

**Data Sources:**
- **ALCE**: Gold passages from `din0s/asqa` (qa_pairs → context)
- **Wikipedia**: `wikimedia/wikipedia` (20231101.en), streamed and chunked

In [6]:
# Cell 6: Load Retrieval Corpus
print("📚 Loading retrieval corpus ...")
all_passages = []

if USE_ALCE_CORPUS:
    print("  Loading ALCE gold passages ...")
    try:
        alce_ds = load_dataset("din0s/asqa", split="dev")
        alce_passages = []
        for record in alce_ds:
            if "qa_pairs" in record:
                for qa in record["qa_pairs"]:
                    if "context" in qa and qa["context"]:
                        ctx = qa["context"]
                        text = ctx if isinstance(ctx, str) else " ".join(ctx) if isinstance(ctx, list) else str(ctx)
                        alce_passages.append({"title": record.get("ambiguous_question","ALCE"), "text": text, "source": "ALCE"})
        all_passages.extend(alce_passages)
        print(f"  ✅ ALCE: {len(alce_passages)} passages")
    except Exception as e:
        print(f"  ⚠️  ALCE failed: {e}")

if USE_WIKIPEDIA:
    print(f"  Loading Wikipedia ({MAX_WIKI_PASSAGES:,} passages) ...")
    try:
        wiki_ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
        wiki_passages = []
        for article in wiki_ds:
            text = article.get("text", "")
            title = article.get("title", "Wikipedia")
            if not text or len(text) < 100: continue
            chunks = [text[i:i+300] for i in range(0, min(len(text), 2000), 300)]
            for chunk in chunks:
                if len(chunk.strip()) > 50:
                    wiki_passages.append({"title": title, "text": chunk.strip(), "source": "Wikipedia"})
            if len(wiki_passages) >= MAX_WIKI_PASSAGES: break
        all_passages.extend(wiki_passages[:MAX_WIKI_PASSAGES])
        print(f"  ✅ Wikipedia: {len(wiki_passages[:MAX_WIKI_PASSAGES])} passages")
    except Exception as e:
        print(f"  ⚠️  Wikipedia failed: {e}")

print(f"\n📊 Total corpus: {len(all_passages):,} passages")
if not all_passages: raise RuntimeError("No passages loaded!")

print("\n📖 Data Sources:")
for s in set(p["source"] for p in all_passages):
    print(f"   - {s}: {sum(1 for p in all_passages if p['source'] == s):,} passages")

📚 Loading retrieval corpus ...
  Loading ALCE gold passages ...
  ✅ ALCE: 3184 passages
  Loading Wikipedia (10,000 passages) ...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

  ✅ Wikipedia: 10000 passages

📊 Total corpus: 13,184 passages

📖 Data Sources:
   - ALCE: 3,184 passages
   - Wikipedia: 10,000 passages


## Cell 7: Build BM25 Index + Load Embeddings
BM25 index is built once and shared across all models.
Embedding model runs on GPU for fast computation.

In [7]:
# Cell 7: Build BM25 + Load Embedding Model
print("🔧 Building BM25 index ...")
retriever = BM25Retriever(all_passages)
print(f"   ✅ BM25 index built ({len(all_passages):,} passages)")

print(f"🧠 Loading embedding model ({EMBEDDING_MODEL}) on {DEVICE} ...")
embedder = Embedder()
print("   ✅ Embedding model ready")

🔧 Building BM25 index ...
   ✅ BM25 index built (13,184 passages)
🧠 Loading embedding model (all-MiniLM-L6-v2) on cuda ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:pcite:  Embedding model loaded on cuda


   ✅ Embedding model ready


## Cell 8: 🏃 Run Attribution Uncertainty Pipeline

**This is the main computation cell.** For each model:
1. Generate answers for all questions
2. Extract claims → generate perturbations (parallel)
3. BM25 retrieve → embed (GPU) → compute U_attr
4. Save per-model results to `results/<model_name>/`

> ⏱️ **Estimated time**: ~1-2 hours per model (free tier with rate limits)

In [ ]:
# Cell 8: Run Pipeline for All Models

def run_pipeline_for_model(model_id, questions):
    """Full attribution uncertainty pipeline for one model."""
    short = MODEL_SHORT.get(model_id, model_id.split("/")[-1])
    model_dir = RESULTS_DIR / short
    model_dir.mkdir(parents=True, exist_ok=True)
    llm = LLMClient(model_id)
    records = []
    jsonl_path = model_dir / "results.jsonl"
    if jsonl_path.exists(): jsonl_path.unlink()

    pbar = tqdm(questions, desc=f"🔄 {short}", unit="q", ncols=95)
    for q_idx, question in enumerate(pbar):
        pbar.set_postfix(claims=len(records))
        try:
            answer = llm.generate_answer(question)
        except Exception as e:
            logger.warning(f"  ❌ Answer failed Q{q_idx+1}: {e}")
            continue

        claims = extract_claims(answer)
        if not claims: continue

        for claim in claims:
            orig_docs = retriever.retrieve(claim, top_k=TOP_K_DOCS)
            orig_emb = embedder.mean_embedding([d["text"] for d in orig_docs])

            # Parallel perturbation generation
            def _gen(c):
                try: return llm.generate_one_perturbation(c)
                except: return c

            with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as pool:
                futures = [pool.submit(_gen, claim) for _ in range(NUM_PERTURBATIONS)]
                perturbations = [f.result() for f in as_completed(futures)]

            sims = []
            for perturb in perturbations:
                p_docs = retriever.retrieve(perturb, top_k=TOP_K_DOCS)
                p_emb = embedder.mean_embedding([d["text"] for d in p_docs])
                sims.append(cosine_sim(orig_emb, p_emb))

            record = {
                "model": model_id, "model_short": short,
                "question": question, "generated_answer": answer,
                "fi": claim, "perturbations": perturbations,
                "similarities": [round(s,6) for s in sims],
                "uncertainty": round(compute_uncertainty(sims), 6),
            }
            records.append(record)
            with open(jsonl_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        time.sleep(3)
    pbar.close()
    return records


def save_model_outputs(records, model_id):
    short = MODEL_SHORT.get(model_id, model_id.split("/")[-1])
    model_dir = RESULTS_DIR / short
    if not records:
        print(f"  ⚠️  No records for {short}")
        return {}
    all_u = [r["uncertainty"] for r in records]
    csv_rows = [{"model": short, "question": r["question"], "claim": r["fi"],
                 "uncertainty": r["uncertainty"], "sim_mean": round(np.mean(r["similarities"]),6),
                 "sim_std": round(np.std(r["similarities"]),6),
                 **{f"sim_{i+1}": s for i,s in enumerate(r["similarities"])}} for r in records]
    pd.DataFrame(csv_rows).to_csv(model_dir / "results.csv", index=False)
    q_u = defaultdict(list)
    for r in records: q_u[r["question"]].append(r["uncertainty"])
    stats = {
        "model_short": short, "model_full": model_id,
        "num_questions": len(q_u), "num_claims": len(records),
        "mean_U_attr": round(statistics.mean(all_u), 6),
        "std_U_attr": round(statistics.stdev(all_u) if len(all_u)>1 else 0, 6),
        "median_U_attr": round(statistics.median(all_u), 6),
        "min_U_attr": round(min(all_u), 6),
        "max_U_attr": round(max(all_u), 6),
        "high_uncertainty_count": sum(1 for u in all_u if u > 0.03),
        "high_uncertainty_pct": round(100*sum(1 for u in all_u if u > 0.03)/len(all_u), 2),
        "low_uncertainty_count": sum(1 for u in all_u if u < 0.005),
        "low_uncertainty_pct": round(100*sum(1 for u in all_u if u < 0.005)/len(all_u), 2),
    }
    with open(model_dir / "stats.json", "w") as f: json.dump(stats, f, indent=2)
    print(f"  📊 {short}: {stats['num_claims']} claims | Mean U={stats['mean_U_attr']:.4f} | Max U={stats['max_U_attr']:.4f} | High%={stats['high_uncertainty_pct']:.1f}%")
    return stats


# ── RUN ALL MODELS ──
print("=" * 65)
print(f"  🚀 STARTING MULTI-MODEL EXPERIMENT")
print(f"  {len(MODELS)} models × {len(questions)} questions")
print("=" * 65)

all_model_stats = []
all_model_records = {}

for m_idx, model_id in enumerate(MODELS):
    short = MODEL_SHORT.get(model_id, model_id)
    print(f"\n{'─'*65}")
    print(f"  Model {m_idx+1}/{len(MODELS)}: {short}")
    print(f"{'─'*65}")
    try:
        records = run_pipeline_for_model(model_id, questions)
        if records:
            stats = save_model_outputs(records, model_id)
            all_model_stats.append(stats)
            all_model_records[short] = records
            print(f"  ✅ {short} complete: {len(records)} claims")
        else:
            print(f"  ❌ {short}: no claims (API rate-limited?)")
    except Exception as e:
        print(f"  ❌ {short} FAILED: {e}")
    if m_idx < len(MODELS) - 1:
        print(f"\n  ⏸️  Pausing 30s between models ...")
        time.sleep(30)

print(f"\n{'='*65}")
print(f"  ✅ EXPERIMENT COMPLETE — {len(all_model_stats)} models succeeded")
print(f"{'='*65}")

  🚀 STARTING MULTI-MODEL EXPERIMENT
  5 models × 40 questions

─────────────────────────────────────────────────────────────────
  Model 1/5: llama-3.3-70b
─────────────────────────────────────────────────────────────────


🔄 llama-3.3-70b:   0%|                                                  | 0/40 [00:00<?, ?q/s]

## Cell 9: 📊 Final Comparison Report
Cross-model comparison with ranking, bar chart, and total uncertainty score.

In [ ]:
# Cell 9: Final Comparison Report
if not all_model_stats:
    print("❌ No models completed. Check API key and rate limits.")
else:
    print("=" * 90)
    print("   ATTRIBUTION UNCERTAINTY — MULTI-MODEL COMPARISON")
    print("   Dataset: ASQA | Retrieval: BM25 | Embeddings: all-MiniLM-L6-v2")
    print("   U_attr = Var(cosine_similarities) across K=5 perturbations")
    print("=" * 90)

    comp_rows = [{
        "Model": s["model_short"], "#Q": s["num_questions"], "#Claims": s["num_claims"],
        "Mean_U": s["mean_U_attr"], "Std_U": s["std_U_attr"],
        "Median_U": s["median_U_attr"], "Max_U": s["max_U_attr"],
        "High_U%": s["high_uncertainty_pct"], "Stable%": s["low_uncertainty_pct"],
    } for s in all_model_stats]

    comp_df = pd.DataFrame(comp_rows).sort_values("Mean_U", ascending=True)
    print("\n📊 MODEL RANKING (lowest uncertainty = most stable attribution):\n")
    print(comp_df.to_string(index=False))

    # Bar chart
    max_u = comp_df["Mean_U"].max()
    print("\n\n📊 UNCERTAINTY BAR CHART:\n")
    for _, row in comp_df.iterrows():
        filled = int((row["Mean_U"] / max_u) * 30) if max_u > 0 else 0
        bar = "█" * filled + "░" * (30 - filled)
        print(f"  {row['Model']:<25s} |{bar}| {row['Mean_U']:.6f}")

    # Total score
    all_means = [s["mean_U_attr"] for s in all_model_stats]
    print(f"\n\n{'─'*60}")
    print(f"  ═══ TOTAL UNCERTAINTY SCORE (across all models) ═══")
    print(f"    Grand Mean U_attr  : {np.mean(all_means):.6f}")
    print(f"    Best  (lowest U)   : {comp_df.iloc[0]['Model']} ({comp_df.iloc[0]['Mean_U']:.6f})")
    print(f"    Worst (highest U)  : {comp_df.iloc[-1]['Model']} ({comp_df.iloc[-1]['Mean_U']:.6f})")
    print(f"{'─'*60}")

    # Save
    comp_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
    with open(RESULTS_DIR / "model_comparison.json", "w") as f:
        json.dump(all_model_stats, f, indent=2)
    print(f"\n  💾 Saved: {RESULTS_DIR}/model_comparison.csv")
    print(f"  💾 Saved: {RESULTS_DIR}/model_comparison.json")

## Cell 10: 📋 Per-Model Detailed View
Expandable breakdown: top uncertain/stable claims per model.

In [ ]:
# Cell 10: Per-Model Detailed View
for short, records in all_model_records.items():
    print(f"\n{'='*70}")
    print(f"  DETAILED VIEW: {short}")
    print(f"{'='*70}")
    all_u = [r["uncertainty"] for r in records]
    q_u = defaultdict(list)
    for r in records: q_u[r["question"]].append(r["uncertainty"])

    print(f"\n  Per-Question Mean Uncertainty (top 10):")
    q_sorted = sorted(q_u.items(), key=lambda kv: -statistics.mean(kv[1]))
    q_max = statistics.mean(q_sorted[0][1]) if q_sorted else 1
    for q, vals in q_sorted[:10]:
        mu = statistics.mean(vals)
        filled = int((mu / q_max) * 20) if q_max > 0 else 0
        bar = "█" * filled + "░" * (20 - filled)
        print(f"    mean_U={mu:.5f}  |{bar}|  {q[:55]}")

    sorted_r = sorted(records, key=lambda r: -r["uncertainty"])
    print(f"\n  Top 3 Most Uncertain Claims:")
    for r in sorted_r[:3]:
        print(f"    U={r['uncertainty']:.6f}  {r['fi'][:70]}")
    print(f"\n  Top 3 Most Stable Claims:")
    for r in sorted_r[-3:]:
        print(f"    U={r['uncertainty']:.6f}  {r['fi'][:70]}")

## Cell 11: 💾 Download Results
List all output files and download them.

In [ ]:
# Cell 11: Download Results
print("📁 All output files:")
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {str(p.relative_to(RESULTS_DIR)):<50s}  ({size:>8,} bytes)")

print(f"\n💡 To download from Colab:")
print(f"   from google.colab import files")
print(f"   files.download(str(RESULTS_DIR / 'model_comparison.csv'))")

# Uncomment to zip and download all results:
# import shutil
# zip_path = shutil.make_archive("/content/pcite_results", "zip", str(RESULTS_DIR))
# from google.colab import files
# files.download(zip_path)